# bulk-water-rdf.ipynb
Computes and plots the O–O radial distribution function g(r) of bulk TIP4P/2005 water at 300 K, and finds its first peak (reference for the one-plate and two-plate densities).

- **Runs on:** PC (needs `/avogadro` mounted)
- **Inputs:** Joe's pure-water run `/avogadro/joe/pure_tip4p_sim/300K/md.nc`, `topol.prmtop`
- **Outputs:** `one-plate/results_pc/bulk-oo-rdf.png`

## Load trajectory
Every 100th frame; selects water oxygens. **Change:** `TRAJ`, `TOP`, `STRIDE`.

In [ ]:
from pathlib import Path
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

OUT = Path.cwd() / "results_pc"
OUT.mkdir(parents=True, exist_ok=True)

TRAJ = "/avogadro/joe/pure_tip4p_sim/300K/md.nc"
TOP = "/avogadro/joe/pure_tip4p_sim/300K/topol.prmtop"
STRIDE = 100                                   # every 100th frame

traj = md.load(TRAJ, top=TOP, stride=STRIDE)
ox = traj.topology.select("name O")

## Compute g(r)
0.2–0.8 nm, bin 0.002 nm; prints the first peak. **Change:** `r_range`, `bin_width`.

In [ ]:
pairs = traj.topology.select_pairs(ox, ox)

r, g = md.compute_rdf(traj, pairs, r_range=(0.2, 0.8), bin_width=0.002)  #nm
r_peak = r[np.argmax(g)] * 10  #A
print(f"First O-O peak at {r_peak:.3f} Å, g(r) = {g.max():.2f}")

## Plot
**Output:** `bulk-oo-rdf.png`.

In [ ]:
r_A = r * 10                    # nm -> Å
i_peak = np.argmax(g)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(r_A, g, color="blue", label="TIP4P/2005 O–O")
ax.axvline(r_A[i_peak], color="gray", ls="--", lw=0.8)
ax.plot(r_A[i_peak], g[i_peak], "ro", label=f"First peak: {r_A[i_peak]:.2f} Å, g(r) = {g[i_peak]:.2f}")
ax.annotate(f"{r_A[i_peak]:.2f} Å", (r_A[i_peak], g[i_peak]),
            textcoords="offset points", xytext=(8, -4))
ax.axhline(1, color="k", ls="--", lw=0.8, label="Bulk density, g(r) = 1")

ax.set_title("Oxygen–oxygen radial distribution function of bulk TIP4P/2005 water, 300 K")
ax.set_xlabel("r (Å)")
ax.set_ylabel(r"$g_{OO}(r)$")
ax.set_ylim(-0.4, 3.5)
ax.set_xlim(2, 8)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "bulk-oo-rdf.png", dpi=800)
plt.show()